In [0]:
-- Databricks SQL does not support IF OBJECT_ID. Use DROP TABLE IF EXISTS.
DROP TABLE IF EXISTS payments;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;

In [0]:
-- 2. CREATE PRODUCTS (The Master List)
CREATE TABLE pooja.bronze_schema.products (
    product_id    INT NOT NULL,
    product_name  VARCHAR(100),
    category      VARCHAR(100),
    price         VARCHAR(30), 
    updated_at    TIMESTAMP NOT NULL,
    PRIMARY KEY (product_id)
);


In [0]:
-- 3. CREATE ORDERS (The Central Hub)
-- This table is "connected" to Products via the product_id FK
CREATE TABLE pooja.bronze_schema.orders (
    order_id      INT NOT NULL,
    customer_id   INT,      
    product_id    INT,      -- The connection to Products
    order_status  VARCHAR(50),      
    order_amount  VARCHAR(30),      
    created_at    TIMESTAMP     NOT NULL,
    updated_at    TIMESTAMP     NOT NULL,
    PRIMARY KEY (order_id)
);


In [0]:
-- 4. CREATE PAYMENTS (Connected to Orders)
-- This table is "connected" to Orders via the order_id FK
CREATE TABLE pooja.bronze_schema.payments (
    payment_id     INT NOT NULL,
    order_id       INT NOT NULL,           -- The connection to Orders
    payment_status VARCHAR(50),  
    paid_amount    VARCHAR(30),  
    processed_at   TIMESTAMP NOT NULL,
    PRIMARY KEY (payment_id)
);

Inserting Data to table 


In [0]:
-- =========================================================
-- INITIAL LOAD : CLEAN + MESSY DATA
-- =========================================================

-- 1. CLEAN UP: Delete in dependency order
DELETE FROM pooja.bronze_schema.payments;
DELETE FROM pooja.bronze_schema.orders;
DELETE FROM pooja.bronze_schema.products;


In [0]:
-- =========================================================
-- 2. PRODUCTS (120 rows: 1001 to 1120)
-- Mix of:
-- - clean product_name/category/price
-- - null product_name
-- - typo category
-- - lowercase category
-- - comma price
-- - dollar sign price
-- - invalid price
-- =========================================================
WITH n AS (
    SELECT ROW_NUMBER() OVER (ORDER BY id) AS rn
    FROM RANGE(1, 121)
)
INSERT INTO pooja.bronze_schema.products (product_id, product_name, category, price, updated_at)
SELECT
    1000 + rn AS product_id,
    CASE
        WHEN rn % 20 = 0 THEN NULL
        WHEN rn % 15 = 0 THEN CONCAT('   Product ', CAST(rn AS STRING), '   ')
        WHEN rn % 11 = 0 THEN CONCAT('PRODUCT-', CAST(rn AS STRING))
        WHEN rn % 9  = 0 THEN CONCAT('Prod_', CAST(rn AS STRING))
        ELSE CONCAT('Product ', CAST(rn AS STRING))
    END AS product_name,
    CASE
        WHEN rn % 18 = 0 THEN 'ELECTRNICS'   -- typo
        WHEN rn % 14 = 0 THEN 'lifestyle'
        WHEN rn % 10 = 0 THEN 'FITNESS'
        WHEN rn % 7  = 0 THEN 'electronics'
        ELSE 'Electronics'
    END AS category,
    CASE
        WHEN rn % 25 = 0 THEN '??'
        WHEN rn % 16 = 0 THEN CONCAT('$', CAST(10 + (rn % 90) AS STRING), '.00')
        WHEN rn % 13 = 0 THEN CONCAT(CAST(10 + (rn % 90) AS STRING), ',00')
        WHEN rn % 8  = 0 THEN CONCAT(' ', CAST(10 + (rn % 90) AS STRING), '.00 ')
        ELSE CONCAT(CAST(10 + (rn % 90) AS STRING), '.00')
    END AS price,
    TIMESTAMP'2026-02-01 09:00:00' + MAKE_INTERVAL(0, 0, 0, 0, 0, rn, 0) AS updated_at
FROM n;

In [0]:
-- =========================================================
-- 3. ORDERS (500 rows: 200001 to 200500)
-- Mix of:
-- - clean rows
-- - null customer_id
-- - null order_status
-- - blank order_status
-- - zero amount
-- - messy amount strings
-- =========================================================
WITH n AS (
    SELECT ROW_NUMBER() OVER (ORDER BY id) AS rn
    FROM RANGE(1, 501)
)
INSERT INTO pooja.bronze_schema.orders (order_id, customer_id, product_id, order_status, order_amount, created_at, updated_at)
SELECT
    200000 + rn AS order_id,
    CASE
        WHEN rn % 40 = 0 THEN NULL
        ELSE 5000 + (rn % 200)
    END AS customer_id,
    1001 + ((rn - 1) % 120) AS product_id,
    CASE
        WHEN rn % 33 = 0 THEN NULL
        WHEN rn % 22 = 0 THEN ''
        WHEN rn % 15 = 0 THEN 'shipped'
        WHEN rn % 9  = 0 THEN 'cancelled'
        ELSE 'PLACED'
    END AS order_status,
    CASE
        WHEN rn % 45 = 0 THEN '0.00'
        WHEN rn % 28 = 0 THEN 'N/A'
        WHEN rn % 17 = 0 THEN CONCAT('$', CAST(50 + (rn % 100) AS STRING), '.00')
        WHEN rn % 12 = 0 THEN CONCAT(CAST(50 + (rn % 100) AS STRING), ',00')
        ELSE CONCAT(CAST(50 + (rn % 100) AS STRING), '.00')
    END AS order_amount,
    TIMESTAMP'2026-02-01 10:00:00' + MAKE_INTERVAL(0, 0, 0, 0, 0, rn, 0) AS created_at,
    TIMESTAMP'2026-02-01 10:00:00' + MAKE_INTERVAL(0, 0, 0, 0, 0, rn, 0) AS updated_at
FROM n;


In [0]:
-- =========================================================
-- 4. PAYMENTS (430 rows: 900001 to 900430)
-- Mix of:
-- - clean rows
-- - null payment_status
-- - zero amount
-- - messy amount strings
-- =========================================================
WITH n AS (
    SELECT ROW_NUMBER() OVER (ORDER BY id) AS rn
    FROM RANGE(1, 431)
)
INSERT INTO pooja.bronze_schema.payments (payment_id, order_id, payment_status, paid_amount, processed_at)
SELECT
    900000 + rn AS payment_id,
    200000 + rn AS order_id,
    CASE
        WHEN rn % 35 = 0 THEN NULL
        WHEN rn % 18 = 0 THEN 'failed'
        WHEN rn % 12 = 0 THEN 'pending'
        ELSE 'SUCCESS'
    END AS payment_status,
    CASE
        WHEN rn % 40 = 0 THEN '0.00'
        WHEN rn % 23 = 0 THEN '??'
        WHEN rn % 16 = 0 THEN '$100.00'
        WHEN rn % 11 = 0 THEN '100,00'
        ELSE '100.00'
    END AS paid_amount,
    TIMESTAMP'2026-02-01 11:00:00' + MAKE_INTERVAL(0, 0, 0, 0, 0, rn, 0) AS processed_at
FROM n;

In [0]:
/* =====================================================
   PRODUCTS — product_id > 1120
   ===================================================== */

INSERT INTO pooja.bronze_schema.products (product_id, product_name, category, price, updated_at)
VALUES
(1121, 'Product 1121', 'ELECTRONICS', '50.00', SYSDATETIME()),
(1122, 'Product 1122', 'LIFESTYLE', '51.00', SYSDATETIME()),
(1123, 'Product 1123', 'FITNESS', '52.00', SYSDATETIME()),
(1124, 'Product 1124', 'ELECTRONICS', '53.00', SYSDATETIME()),
(1125, 'Product 1125', 'LIFESTYLE', '54.00', SYSDATETIME()),
(1126, 'Product 1126', 'FITNESS', '55.00', SYSDATETIME()),
(1127, 'Product 1127', 'ELECTRONICS', '56.00', SYSDATETIME()),
(1128, 'Product 1128', 'LIFESTYLE', '57.00', SYSDATETIME()),
(1129, 'Product 1129', 'FITNESS', '58.00', SYSDATETIME()),
(1130, 'Product 1130', 'ELECTRONICS', '59.00', SYSDATETIME());

In [0]:
/* =====================================================
   PRODUCTS — product_id > 1120
   ===================================================== */

INSERT INTO pooja.bronze_schema.products (product_id, product_name, category, price, updated_at)
VALUES
(1121, 'Product 1121', 'ELECTRONICS', '50.00', SYSDATETIME()),
(1122, 'Product 1122', 'LIFESTYLE', '51.00', SYSDATETIME()),
(1123, 'Product 1123', 'FITNESS', '52.00', SYSDATETIME()),
(1124, 'Product 1124', 'ELECTRONICS', '53.00', SYSDATETIME()),
(1125, 'Product 1125', 'LIFESTYLE', '54.00', SYSDATETIME()),
(1126, 'Product 1126', 'FITNESS', '55.00', SYSDATETIME()),
(1127, 'Product 1127', 'ELECTRONICS', '56.00', SYSDATETIME()),
(1128, 'Product 1128', 'LIFESTYLE', '57.00', SYSDATETIME()),
(1129, 'Product 1129', 'FITNESS', '58.00', SYSDATETIME()),
(1130, 'Product 1130', 'ELECTRONICS', '59.00', SYSDATETIME());


/* =====================================================
   ORDERS — order_id > 200500
   ===================================================== */

INSERT INTO pooja.bronze_schema.orders
(order_id, customer_id, product_id, order_status, order_amount, created_at, updated_at)
VALUES
(200501, 6001, 1121, 'PLACED', '100.00', SYSDATETIME(), SYSDATETIME()),
(200502, 6002, 1122, 'SHIPPED', '101.00', SYSDATETIME(), SYSDATETIME()),
(200503, 6003, 1123, 'PLACED', '102.00', SYSDATETIME(), SYSDATETIME()),
(200504, 6004, 1124, 'PLACED', '103.00', SYSDATETIME(), SYSDATETIME()),
(200505, 6005, 1125, 'SHIPPED', '104.00', SYSDATETIME(), SYSDATETIME()),
(200506, 6006, 1126, 'PLACED', '105.00', SYSDATETIME(), SYSDATETIME()),
(200507, 6007, 1127, 'SHIPPED', '106.00', SYSDATETIME(), SYSDATETIME()),
(200508, 6008, 1128, 'PLACED', '107.00', SYSDATETIME(), SYSDATETIME()),
(200509, 6009, 1129, 'PLACED', '108.00', SYSDATETIME(), SYSDATETIME()),
(200510, 6010, 1130, 'SHIPPED', '109.00', SYSDATETIME(), SYSDATETIME());


/* =====================================================
   PAYMENTS — payment_id > 900430
   ===================================================== */

INSERT INTO pooja.bronze_schema.payments
(payment_id, order_id, payment_status, paid_amount, processed_at)
VALUES
(900431, 200501, 'SUCCESS', '100.00', SYSDATETIME()),
(900432, 200502, 'SUCCESS', '101.00', SYSDATETIME()),
(900433, 200503, 'SUCCESS', '102.00', SYSDATETIME()),
(900434, 200504, 'SUCCESS', '103.00', SYSDATETIME()),
(900435, 200505, 'SUCCESS', '104.00', SYSDATETIME()),
(900436, 200506, 'SUCCESS', '105.00', SYSDATETIME()),
(900437, 200507, 'SUCCESS', '106.00', SYSDATETIME()),
(900438, 200508, 'SUCCESS', '107.00', SYSDATETIME()),
(900439, 200509, 'SUCCESS', '108.00', SYSDATETIME()),
(900440, 200510, 'SUCCESS', '109.00', SYSDATETIME());